# Superstore Sales: Exploratory Data Analysis
Dataset: Sample - Superstore (Kaggle, CC0). Goal: clean and explore the orders with pandas, and create five seaborn charts that give the e-commerce manager a visual summary.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid")
df = pd.read_csv("Sample - Superstore.csv", encoding="latin-1")
df.head()

## 1. Inspect the data

In [ ]:
print(df.shape)
df.info()
print("Missing values:", df.isna().sum().sum())
print("Duplicate rows:", df.drop(columns="Row ID").duplicated().sum())
print("Postal codes with 4 digits:", (df["Postal Code"] < 10000).sum())

## 2. Clean the data

In [ ]:
df = df.drop_duplicates(subset=df.columns.drop("Row ID"))
df["Order Date"] = pd.to_datetime(df["Order Date"], format="%m/%d/%Y")
df["Ship Date"] = pd.to_datetime(df["Ship Date"], format="%m/%d/%Y")
df["Postal Code"] = df["Postal Code"].astype(str).str.zfill(5)
df["Unit Price"] = df["Sales"] / df["Quantity"]
print(df.shape)
print(df["Order Date"].min(), "to", df["Order Date"].max())
df[["Order Date", "Ship Date", "Postal Code", "Unit Price"]].dtypes

In [ ]:
df[["Sales", "Quantity", "Discount", "Profit", "Unit Price"]].describe().round(2)

## 3. Visualizations

In [ ]:
plt.figure(figsize=(8, 5))
ax = sns.barplot(data=df, x="Category", y="Sales", estimator="sum", errorbar=None)
ax.set(title="Total Sales by Category", xlabel="Category", ylabel="Total sales (USD)")
plt.savefig("01_bar_sales_by_category.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
monthly = df.resample("MS", on="Order Date")["Sales"].sum().reset_index()
plt.figure(figsize=(12, 5))
ax = sns.lineplot(data=monthly, x="Order Date", y="Sales", marker="o")
ax.set(title="Monthly Revenue, 2014 to 2017", xlabel="Month", ylabel="Revenue (USD)")
plt.savefig("02_line_monthly_revenue.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
order_values = df.groupby("Order ID")["Sales"].sum()
plt.figure(figsize=(8, 5))
ax = sns.histplot(order_values, bins=40, log_scale=True)
ax.set(title="Distribution of Order Values", xlabel="Order value (USD, log scale)", ylabel="Number of orders")
plt.savefig("03_hist_order_values.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
ax = sns.scatterplot(data=df, x="Unit Price", y="Quantity", alpha=0.3)
ax.set(xscale="log", title="Unit Price vs. Quantity", xlabel="Unit price (USD, log scale)", ylabel="Quantity per order line")
plt.savefig("04_scatter_price_vs_quantity.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
numeric = ["Sales", "Quantity", "Discount", "Profit", "Unit Price"]
plt.figure(figsize=(7, 5))
ax = sns.heatmap(df[numeric].corr(), annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1)
ax.set_title("Correlation Between Numeric Variables")
plt.savefig("05_heatmap_correlations.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Observations
- **Data:** 9,994 rows and no missing values. We removed 1 duplicate, fixed the dates and the postal codes, and added Unit Price.
- **Category:** Technology has the highest sales, about $836K.
- **Trend:** revenue grows from about $484K in 2014 to $733K in 2017, with peaks at the end of each year.
- **Orders:** the median order is about $152, but a few orders are over $5,000.
- **Price vs. quantity:** no relationship (correlation 0.00).
- **Discount:** higher discounts go with lower profit (-0.22).